# Récupérer les données géographiques (pays, villes)



In [11]:
#chargement des modules
import pandas as pd
import datetime
import os
import json
from io import StringIO
import numpy as np
import plotly.express as px
import re

In [3]:
import s3fs #pour connecter au bucket

In [4]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [5]:
file_path = BUCKET_OUT + "/" + "Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.jsonl"
print(file_path)
list_id_pub = []
datas =[]
with fs.open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            data = json.loads(line)
            # Extract specific fields
            ids = data.get('all_ids')
            list_id_pub.append(ids[0])
            datas.append(data)
            
        except json.JSONDecodeError:
            continue  # Skip invalid lines

aluneau/Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.jsonl


In [7]:
datas[0]

{'all_ids': ['halhal-03933378', 'naturalf8f5d55c1bafa6553b6502f859b457e1'],
 'authors': [{'hal_docid': '28862-176081',
   'first_name': 'Jérôme',
   'last_name': 'Livet',
   'full_name': 'Jérôme Livet',
   'id_hal_i': '176081',
   'id_hal_s': 'jerome-livet',
   'orcid': '0000-0002-1785-776X',
   'author_position': 1,
   'role': 'aut',
   'affiliations': [{'detected_countries': ['fr'],
     'name': "Institut national de recherches archéologiques préventives, Inrap, 121 rue d'Alésia75014 Paris, France",
     'hal_docid': '301475',
     'country': 'France',
     'structId': '301475'}],
   'datasource': 'hal'},
  {'hal_docid': '7952-15426',
   'first_name': 'François',
   'last_name': 'Capron',
   'full_name': 'François Capron',
   'id_hal_i': '15426',
   'idref': '076914968',
   'id_hal_s': 'francois-capron',
   'author_position': 2,
   'role': 'aut',
   'affiliations': [{'detected_countries': ['fr'],
     'name': "Institut national de recherches archéologiques préventives, Inrap, 121 rue

In [15]:
authors_halid_s = []
authors_halid_i = []
authors_fullname = []
authors_orcid = []
structures_id = []
structures_country = []
structures_address = []
hal_class_code = []
bso_class = []
pub_year = []
pub_date = []
pub_hal_id = []
is_oa = []
list_ids = []

for n, x in enumerate(list_id_pub):
    
    dict_pub = datas[n]
    if 'authors' in dict_pub:
        #print(n, x)
        list_author = []
        list_author_id_s = []
        list_author_id_i = []
        list_author_orcid = []
        list_structures_id = []
        list_country = []
        list_address = []
        list_keyword = []
        list_ids.append(x)
        
        for a in dict_pub['authors']:
            #print(a)
            
            if  'full_name' in a:
                list_author.append(a['full_name'])
            else:
                list_author.append('no full_name')
            if 'id_hal_s' in a:
                list_author_id_s.append(a['id_hal_s'])
            else:
                list_author_id_s.append('no_id_hal_s')
            if 'id_hal_i'  in a:
                list_author_id_i.append(a['id_hal_i'])
            else:
                list_author_id_i.append('no_id_hal_i')
            if 'orcid' in a:
                list_author_orcid.append(a['orcid'])
            else:
                list_author_orcid.append('no orcid')
            if "affiliations" in a and len(a["affiliations"]) > 0 :
                dict_affil = a["affiliations"][0]
                if "country" in dict_affil:
                    list_country.append(dict_affil["country"])
                else:
                    list_country.append("no country")
                if "name" in dict_affil:
                    list_address.append(dict_affil["name"])
                else:
                    list_address.append("no address")
            else:
                list_country.append("no country")
                list_address.append("no address")
    
        authors_halid_s.append("|".join(list_author_id_s))
        authors_halid_i.append("|".join(list_author_id_i))
        authors_fullname.append("|".join(list_author))
        authors_orcid.append("|".join(list_author_orcid))
        structures_country.append("|".join(list_country))
        structures_address.append("|".join(list_address))

        if 'hal' in x:
            pub_hal_id.append(x[3:])
        else:
            pub_hal_id.append(np.nan)
        if 'hal_classification' in dict_pub:
            hal_classification = [h['code'] for h in dict_pub['hal_classification']]
            hal_class_code.append('|'.join(hal_classification))
        else:
            hal_class_code.append('no hal_classification')
        bso_class.append(dict_pub['bso_classification'])
        if 'keywords' in dict_pub:
            for kw in dict_pub["keywords"]:
                list_keyword.append(kw["keyword"])
        else:
            list_keyword.append('no keywords')
        affiliations_i = [inst for inst in dict_pub['bso_local_affiliations']  if re.search(r'[a-z]', inst) is None ]
        structures_id.append('|'.join(affiliations_i))
        #pub_date.append(dict_pub['published_date'])
        pub_year.append(dict_pub['year'])

        for y in dict_pub['oa_details']:
            if y == "2024Q4":
                dict_oa = dict_pub['oa_details'][y]
                #published_date.append(dict_pub['external_ids']['published_date'])
                is_oa.append(dict_oa["is_oa"])
    else:
        pass
        
    

In [20]:
data_columns = {
    "id": list_ids,
    'hal_id': pub_hal_id,
    "pub_year" : pub_year,
    'authors' : authors_fullname,
    'authhalid_s': authors_halid_s,
    'authhalid_i':authors_halid_i,
    'orcid': authors_orcid,
    'struchal_id': structures_id,
    'struc_country': structures_country,
    'struc_address': structures_address,
    'hal_classif' : hal_class_code,
    'bso_classif' : bso_class,
    'is_oa': is_oa
}

df0 = pd.DataFrame(data= data_columns)


In [22]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/publis/2026-02-18_list_of_publi_from_bso.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)